<a href="https://colab.research.google.com/github/carols-rodrigues/fastqc_script_GA055/blob/main/fastqc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%bash

# Pacotes necessários para realizar a análise

apt-get -qq update

# Java
apt-get install -y default-jre

# FastQC
apt-get install -y fastqc

# Cutadapt
pip install cutadapt

# Trim Galore
curl https://sh.rustup.rs -sSf | sh -s -- -y
source "$HOME/.cargo/env"
cargo install trim-galore
trim_galore --version

# SRA Toolkit
apt-get install -y sra-toolkit

# MultiQC
pip install multiqc

# Compressão (opcional)
apt-get install -y pigz

Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  at-spi2-core default-jre-headless fonts-dejavu-core fonts-dejavu-extra
  gsettings-desktop-schemas libatk-bridge2.0-0 libatk-wrapper-java
  libatk-wrapper-java-jni libatk1.0-0 libatk1.0-data libatspi2.0-0
  libxcomposite1 libxtst6 libxxf86dga1 openjdk-11-jre openjdk-11-jre-headless
  session-migration x11-utils
Suggested packages:
  libnss-mdns fonts-ipafont-gothic fonts-ipafont-mincho fonts-wqy-microhei
  | fonts-wqy-zenhei fonts-indic mesa-utils
The following NEW packages will be installed:
  at-spi2-core default-jre default-jre-headless fonts-dejavu-core
  fonts-dejavu-extra gsettings-desktop-schemas libatk-bridge2.0-0
  libatk-wrapper-java libatk-wrapper-java-jni libatk1.0-0 libatk1.0-data
  libatspi2.0-0 libxcomposite1 libxtst6 libxxf86dga1 openjdk-11-jre
  openjdk-11-jre-headless session-migration x11-utils
0 upgraded, 19 newly installed, 0 to r

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
bash: line 14: cargo: command not found


In [23]:
%%writefile pipeline.sh
#!/bin/bash

set -euo pipefail

###############################################################################
# CONFIGURAÇÃO DO PATH
###############################################################################

# Trim Galore está instalado neste diretório
export PATH="/root/.cargo/bin:$PATH"

###############################################################################
# DIRETÓRIOS
###############################################################################

mkdir -p input
mkdir -p fastq
mkdir -p trimmed
mkdir -p qc
mkdir -p logs
mkdir -p tmp

###############################################################################
# CONFIGURAÇÃO DO LOG
###############################################################################

LOG_FILE="logs/pipeline_$(date +%Y%m%d_%H%M%S).log"

# Mostra a saída no Colab e salva simultaneamente no log
exec > >(tee -a "$LOG_FILE") 2>&1

###############################################################################
# FUNÇÕES DE LOG
###############################################################################

log_info() {
    echo "[$(date '+%Y-%m-%d %H:%M:%S')] [INFO] $1"
}

log_ok() {
    echo "[$(date '+%Y-%m-%d %H:%M:%S')] [OK] $1"
}

log_error() {
    echo "[$(date '+%Y-%m-%d %H:%M:%S')] [ERROR] $1"
}

###############################################################################
# TRATAMENTO DE ERROS
###############################################################################

trap 'log_error "Pipeline interrompido na linha $LINENO. Comando: $BASH_COMMAND"' ERR

###############################################################################
# AMOSTRAS
###############################################################################

SAMPLES=(
    SRR5221957
    SRR5221958
    SRR5221961
    SRR5221962
)

###############################################################################
# CONFIGURAÇÕES
###############################################################################

THREADS=$(nproc)
BASE_DIR=$(pwd)

###############################################################################
# INÍCIO DO PIPELINE
###############################################################################

echo
echo "======================================================"
echo "                 PIPELINE INICIADO"
echo "======================================================"

log_info "Data: $(date)"
log_info "CPUs disponíveis: $THREADS"
log_info "Diretório: $BASE_DIR"
log_info "Arquivo de log: $LOG_FILE"

echo "======================================================"

###############################################################################
# VERIFICAÇÃO DAS FERRAMENTAS
###############################################################################

log_info "Verificando ferramentas..."

if command -v prefetch >/dev/null 2>&1; then
    log_ok "prefetch encontrado: $(command -v prefetch)"
else
    log_error "prefetch não encontrado."
    exit 1
fi

if command -v fasterq-dump >/dev/null 2>&1; then
    log_ok "fasterq-dump encontrado: $(command -v fasterq-dump)"
else
    log_error "fasterq-dump não encontrado."
    exit 1
fi

if command -v fastqc >/dev/null 2>&1; then
    log_ok "FastQC encontrado: $(command -v fastqc)"
else
    log_error "FastQC não encontrado."
    exit 1
fi

if command -v multiqc >/dev/null 2>&1; then
    log_ok "MultiQC encontrado: $(command -v multiqc)"
else
    log_error "MultiQC não encontrado."
    exit 1
fi

if command -v trim_galore >/dev/null 2>&1; then
    log_ok "Trim Galore encontrado: $(command -v trim_galore)"
else
    log_error "Trim Galore não encontrado."
    exit 1
fi

echo

###############################################################################
# ESPAÇO DISPONÍVEL
###############################################################################

log_info "Espaço disponível em /content:"

df -h /content

echo

###############################################################################
# DOWNLOAD + FASTQ + FASTQC
###############################################################################

for SAMPLE in "${SAMPLES[@]}"
do

    echo
    echo "======================================================"
    log_info "Processando amostra: $SAMPLE"
    echo "======================================================"

    mkdir -p "qc/$SAMPLE"

    ###########################################################################
    # DOWNLOAD DO SRA
    ###########################################################################

    if [ -f "input/$SAMPLE/$SAMPLE.sra" ]; then

        log_info "[$SAMPLE] Arquivo SRA já existe. Pulando download."

    else

        START=$(date +%s)

        log_info "[$SAMPLE] Iniciando download do SRA..."

        prefetch "$SAMPLE" \
            --output-directory input

        END=$(date +%s)

        log_ok "[$SAMPLE] Download concluído em $((END-START)) segundos."

    fi

    ###########################################################################
    # CONVERSÃO SRA → FASTQ
    ###########################################################################

    if [ -f "fastq/${SAMPLE}.fastq" ]; then

        log_info "[$SAMPLE] FASTQ já existe. Pulando fasterq-dump."

    else

        START=$(date +%s)

        log_info "[$SAMPLE] Convertendo SRA para FASTQ..."

        fasterq-dump \
            "input/$SAMPLE/$SAMPLE.sra" \
            --threads "$THREADS" \
            --temp tmp \
            --outdir fastq

        END=$(date +%s)

        log_ok "[$SAMPLE] Conversão concluída em $((END-START)) segundos."

    fi

    ###########################################################################
    # FASTQC
    ###########################################################################

    if [ -f "qc/$SAMPLE/${SAMPLE}_fastqc.html" ]; then

        log_info "[$SAMPLE] FastQC já foi executado. Pulando."

    else

        START=$(date +%s)

        log_info "[$SAMPLE] Executando FastQC..."

        fastqc \
            "fastq/${SAMPLE}.fastq" \
            -t "$THREADS" \
            -o "qc/$SAMPLE"

        END=$(date +%s)

        log_ok "[$SAMPLE] FastQC concluído em $((END-START)) segundos."

    fi

    log_ok "[$SAMPLE] Etapas iniciais concluídas."

done

###############################################################################
# MULTIQC
###############################################################################

echo
echo "======================================================"
log_info "Executando MultiQC..."
echo "======================================================"

START=$(date +%s)

multiqc \
    qc \
    --outdir qc

END=$(date +%s)

log_ok "MultiQC concluído em $((END-START)) segundos."

###############################################################################
# TRIM GALORE
###############################################################################

echo
echo "======================================================"
log_info "Iniciando Trim Galore..."
echo "======================================================"

for SAMPLE in "${SAMPLES[@]}"
do

    START=$(date +%s)

    if [ -f "trimmed/${SAMPLE}_trimmed.fq" ]; then

        log_info "[$SAMPLE] Arquivo trimmed já existe. Pulando Trim Galore."

    else

        log_info "[$SAMPLE] Executando Trim Galore..."

        trim_galore \
            "fastq/${SAMPLE}.fastq" \
            --output_dir trimmed

        END=$(date +%s)

        log_ok "[$SAMPLE] Trim Galore concluído em $((END-START)) segundos."

    fi

done

###############################################################################
# FASTQC PÓS-TRIMAGEM
###############################################################################

mkdir -p qc/fastqc_trimmed

echo
echo "======================================================"
log_info "Iniciando FastQC pós-trimagem..."
echo "======================================================"

for SAMPLE in "${SAMPLES[@]}"
do

    if [ -f "qc/fastqc_trimmed/${SAMPLE}_trimmed_fastqc.html" ]; then

        log_info "[$SAMPLE] FastQC pós-trimagem já existe. Pulando."

    else

        START=$(date +%s)

        log_info "[$SAMPLE] Executando FastQC pós-trimagem..."

        fastqc \
            "trimmed/${SAMPLE}_trimmed.fq" \
            -t "$THREADS" \
            -o qc/fastqc_trimmed

        END=$(date +%s)

        log_ok "[$SAMPLE] FastQC pós-trimagem concluído em $((END-START)) segundos."

    fi

done

###############################################################################
# MULTIQC FINAL
###############################################################################

echo
echo "======================================================"
log_info "Executando MultiQC pós-trimagem..."
echo "======================================================"

START=$(date +%s)

multiqc \
    qc/fastqc_trimmed \
    --outdir qc \
    --filename multireport_trimmed

END=$(date +%s)

log_ok "MultiQC pós-trimagem concluído em $((END-START)) segundos."

###############################################################################
# FINAL
###############################################################################

echo
echo "======================================================"
echo "                PIPELINE FINALIZADO"
echo "======================================================"

log_ok "Data de término: $(date)"
log_ok "Log completo: $LOG_FILE"

echo "======================================================"

Overwriting pipeline.sh


In [24]:
!chmod +x pipeline.sh
# está dando permissão de execução para o arquivo

In [25]:
# Executar o arquivo
!./pipeline.sh


                 PIPELINE INICIADO
[2026-08-18 17:03:45] [INFO] Data: Tue Aug 18 05:03:45 PM UTC 2026
[2026-08-18 17:03:45] [INFO] CPUs disponíveis: 2
[2026-08-18 17:03:45] [INFO] Diretório: /content
[2026-08-18 17:03:45] [INFO] Arquivo de log: logs/pipeline_20260818_170345.log
[2026-08-18 17:03:45] [INFO] Verificando ferramentas...
[2026-08-18 17:03:45] [OK] prefetch encontrado: /usr/bin/prefetch
[2026-08-18 17:03:45] [OK] fasterq-dump encontrado: /usr/bin/fasterq-dump
[2026-08-18 17:03:45] [OK] FastQC encontrado: /usr/bin/fastqc
[2026-08-18 17:03:45] [OK] MultiQC encontrado: /usr/local/bin/multiqc
[2026-08-18 17:03:45] [OK] Trim Galore encontrado: /root/.cargo/bin/trim_galore

[2026-08-18 17:03:45] [INFO] Espaço disponível em /content:
Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   35G   74G  33% /


[2026-08-18 17:03:45] [INFO] Processando amostra: SRR5221957
[2026-08-18 17:03:45] [INFO] [SRR5221957] Arquivo SRA já existe. Pulando download.
[2026-08-18 17:0

In [26]:
!find . -name "multiqc_report.html"

./qc/multiqc_report.html


In [27]:
from google.colab import files

files.download("qc/multiqc_report.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [28]:
!find . -name "*.html"

./qc/multireport_trimmed.html
./qc/SRR5221958/SRR5221958_fastqc.html
./qc/multiqc_report.html
./qc/SRR5221961/SRR5221961_fastqc.html
./qc/fastqc_trimmed/SRR5221958_trimmed_fastqc.html
./qc/fastqc_trimmed/SRR5221962_trimmed_fastqc.html
./qc/fastqc_trimmed/SRR5221957_trimmed_fastqc.html
./qc/fastqc_trimmed/SRR5221961_trimmed_fastqc.html
./qc/SRR5221957/SRR5221957_fastqc.html
./qc/multiqc_report_1.html
./qc/SRR5221962/SRR5221962_fastqc.html


In [35]:
!ls logs

pipeline_20260818_163745.log  pipeline_20260818_170345.log
pipeline_20260818_165650.log


In [36]:
from google.colab import files

files.download('logs/pipeline_20260818_163745.log')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [33]:
from google.colab import files

files.download("qc/SRR5221958/SRR5221958_fastqc.html")
files.download("qc/SRR5221961/SRR5221961_fastqc.html")
files.download("qc/SRR5221957/SRR5221957_fastqc.html")
files.download("qc/SRR5221962/SRR5221962_fastqc.html")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>